<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Una Red de Sensores del Aire que Hay que Modelar con Honestidad 🌫️
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 06
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/homeworks/06_Modelamiento_Estadistico_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 06: Modelamiento Estadístico** ([README del módulo](../06%20-%20Modelamiento%20Estadistico/README.md)) con un problema **nuevo**: una **red ficticia de sensores de calidad del aire** (estaciones de distinto tipo, conteos de excedencias, predios monitoreados y bancos de variables que son puro ruido).

Al terminar serás capaz de:

1. **Representar categorías sin inventar supuestos:** *one-hot* desde NumPy, la trampa de la variable ficticia, el **orden falso** del *label encoding* medido con validación cruzada y la **fuga** del *target encoding* ingenuo frente al *out-of-fold*.
2. **Modelar conteos y resistir atípicos:** el **IRLS** de Poisson con *offset* desde cero, la **sobredispersión** (parámetro $\hat\phi$, errores cuasi-Poisson) y la pendiente de **Theil-Sen** frente a MCO contaminado.
3. **Evaluar con honestidad:** $R^2$, $R^2$ ajustado y RMSE; sobreajuste con variables de ruido; AIC/BIC; ***k-fold*** y el atajo de ***leave-one-out*** con la matriz sombrero; y la **fuga al seleccionar variables**.
4. **Integrar `scikit-learn` y `statsmodels`:** un *pipeline* con `ColumnTransformer` que sobrevive a categorías nuevas y datos faltantes, un IC *bootstrap* del error de prueba y el mismo modelo ajustado en ambas bibliotecas.

> 📚 **Repasa si lo necesitas:** [Representación de datos](../06%20-%20Modelamiento%20Estadistico/00_Representacion_de_Datos_OneHot_y_Label_Encoding.ipynb) · [GLM y métodos no paramétricos](../06%20-%20Modelamiento%20Estadistico/01_Modelos_Lineales_Generalizados_y_No_Parametricos.ipynb) · [Evaluación del modelo](../06%20-%20Modelamiento%20Estadistico/02_Evaluacion_del_Modelo_R2_MSE_y_Validacion_Cruzada.ipynb) · [Scikit-Learn y Statsmodels](../06%20-%20Modelamiento%20Estadistico/03_Ejercicio_Practico_Scikit_Learn_y_Statsmodels.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos**: los datos son **sintéticos e inventados**, con semilla fija (describen el mecanismo, no una red real). Bibliotecas: `numpy`, `pandas`, `scipy`, `statsmodels` y `scikit-learn`. Todo el cuaderno corre en pocos segundos.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import math

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEMILLA = 21


def generar_tipos(semilla=SEMILLA, n=300):
    """PM2.5 (µg/m³, ficticio) por tipo de estación: el efecto NO es monótono en el orden alfabético."""
    rng = np.random.default_rng(semilla)
    tipo = rng.choice(["industrial", "rural", "urbana", "vial"], n)
    ef = {"industrial": 20.0, "rural": 0.0, "urbana": 10.0, "vial": 30.0}
    return pd.DataFrame({"tipo": tipo, "pm": np.round(35 + np.array([ef[t] for t in tipo]) + rng.normal(0, 6, n), 1)})


def generar_te(semilla=SEMILLA, n=400, k=120):
    """Una variable respuesta de ruido puro y una categoría de alta cardinalidad (120 estaciones) SIN relación con ella."""
    rng = np.random.default_rng(semilla + 1)
    return pd.DataFrame({"estacion": [f"E{i:03d}" for i in rng.integers(0, k, n)], "y": rng.normal(0, 1, n)})


def generar_conteos(semilla=SEMILLA, n=200):
    """Excedencias del límite diario por estación-mes: Poisson mezclada con gamma (sobredispersión) y exposición `dias`."""
    rng = np.random.default_rng(semilla + 2)
    tipo = rng.choice(["rural", "urbana", "vial"], n)
    traf = rng.normal(0, 1, n)
    dias = rng.choice([28, 30, 31], n)
    eta = -1.2 + 0.35 * traf + np.where(tipo == "urbana", 0.4, 0) + np.where(tipo == "vial", 0.8, 0)
    mu = dias * np.exp(eta) * rng.gamma(2.0, 0.5, n)
    return pd.DataFrame({"tipo": tipo, "trafico": np.round(traf, 3), "dias": dias, "excedencias": rng.poisson(mu)})


def generar_contaminada(semilla=SEMILLA, n=40):
    """Relación lineal (pendiente verdadera 2) con 5 mediciones contaminadas por un sensor descalibrado."""
    rng = np.random.default_rng(semilla + 3)
    x = np.round(rng.uniform(0, 10, n), 2)
    yv = 3 + 2 * x + rng.normal(0, 1, n)
    yv[rng.choice(n, 5, replace=False)] += 35
    return x, np.round(yv, 2)


def generar_sobreajuste(semilla=SEMILLA, n=80, p=40):
    """Respuesta que solo depende de las 2 primeras columnas; las otras 38 son ruido."""
    rng = np.random.default_rng(semilla + 4)
    X = rng.normal(size=(n, p))
    return X, 2 + 3 * X[:, 0] - 2 * X[:, 1] + rng.normal(0, 3, n)


def generar_ruido_puro(semilla=SEMILLA, n=60, p=150):
    """150 variables y una respuesta, todas independientes: no hay nada que aprender."""
    rng = np.random.default_rng(semilla + 5)
    return rng.normal(size=(n, p)), rng.normal(size=n)


def generar_predios(semilla=SEMILLA, n=400):
    """Predios monitoreados: 3 numéricas (altitud con ~8 % de faltantes), 2 categóricas y un índice de calidad."""
    rng = np.random.default_rng(semilla + 6)
    area = np.round(rng.lognormal(1.0, 0.6, n), 2)
    alt = np.round(rng.uniform(2400, 3200, n))
    pen = np.round(rng.uniform(0, 35, n), 1)
    uso = rng.choice(["cultivo", "pasto", "bosque"], n, p=[.5, .3, .2])
    ver = rng.choice(["Alisos", "Cerro", "Chorro", "Palmar", "Sabana", "Tejar"], n)
    efv = dict(zip(["Alisos", "Cerro", "Chorro", "Palmar", "Sabana", "Tejar"], [0, 2, -1, 3, -2, 1]))
    efu = {"cultivo": 0, "pasto": -5, "bosque": -9}
    y = (40 + 4 * (area - area.mean()) / area.std() + 3 * (alt - alt.mean()) / alt.std() - 2 * (pen - pen.mean()) / pen.std()
         + np.array([efu[u] for u in uso]) + np.array([efv[v] for v in ver]) + rng.normal(0, 4, n))
    alt = alt.astype(float)
    alt[rng.random(n) < 0.08] = np.nan
    return pd.DataFrame({"area": area, "altitud": alt, "pendiente": pen, "uso": uso, "vereda": ver, "indice": np.round(y, 2)})


print("Listo: tipos", generar_tipos().shape, "| conteos", generar_conteos().shape, "| predios", generar_predios().shape)

---
## Reto 1: Traducir categorías sin inventar un orden 🔤

**Contexto:** `aire` registra el PM2.5 (`pm`) de 300 estaciones de cuatro `tipo`s. En orden alfabético los tipos quedan `industrial`, `rural`, `urbana`, `vial`, y su efecto verdadero (20, 0, 10, 30) **no** sube ni baja de forma ordenada. Además, `ruido` trae una respuesta sin relación alguna con la categoría `estacion` (120 niveles), ideal para ver la fuga del *target encoding*.

**Enunciado**
1. `onehot(valores, categorias, quitar_primera)`: matriz `ndarray` de 0/1 (tipo `float`) con una columna por categoría (en el orden de `categorias`), construida con NumPy (comparaciones, **sin** `get_dummies`); si `quitar_primera`, elimina la primera columna. Obtén `D_completa` y `D_ref` para `aire["tipo"]`.
2. `rango_completa`: rango de la matriz `[1 | D_completa]` (columna de unos más todas las indicadoras) y `rango_ref`: rango de `[1 | D_ref]` (`np.linalg.matrix_rank`).
3. `cod_label`: el *label encoding* alfabético (0 a 3) como columna `(300, 1)`. Con `KFold(5, shuffle=True, random_state=0)` y `LinearRegression`, calcula `r2_label` y `r2_onehot` (R² medio de validación cruzada con `cod_label` y con `D_ref`).
4. **Target encoding.** `te_ingenuo(cat_ent, y_ent, cat_nuevas)`: media de `y_ent` por categoría; categorías no vistas reciben la media global de `y_ent`. `te_oof(cat_ent, y_ent, k)`: codificación *out-of-fold* (la fila $i$ se codifica con las filas de **otros** pliegues; el pliegue de la fila $i$ es `i % k`). Con las primeras 300 filas de `ruido` como entrenamiento y las últimas 100 como prueba, ajusta una regresión lineal de `y` sobre cada codificación (en prueba se codifica con `te_ingenuo` usando **todo** el entrenamiento) y guarda `r2_ent_ing`, `r2_prb_ing`, `r2_ent_oof`, `r2_prb_oof`.

**Criterios de aceptación:** tu *one-hot* coincide con `pd.get_dummies`; `rango_completa` es **4 pese a tener 5 columnas** y `rango_ref` es 4; el *label encoding* rinde mucho peor que el *one-hot* en validación cruzada; y el *target encoding* ingenuo muestra un $R^2$ de entrenamiento alto que **se desploma** en prueba, mientras el *out-of-fold* no se hace ilusiones.

**Puntaje sugerido:** 22 puntos.

In [ ]:
aire = generar_tipos()
CATS = np.array(sorted(aire["tipo"].unique()))
ruido = generar_te()
cv5 = KFold(5, shuffle=True, random_state=0)


def onehot(valores, categorias, quitar_primera):
    # TODO: matriz 0/1 (float) con una columna por categoría; si quitar_primera, sin la primera columna
    return None


def te_ingenuo(cat_ent, y_ent, cat_nuevas):
    # TODO: media de y_ent por categoría (global para categorías no vistas)
    return None


def te_oof(cat_ent, y_ent, k=5):
    # TODO: codificación out-of-fold; el pliegue de la fila i es i % k
    return None


# TODO: completa
D_completa = D_ref = None
rango_completa = rango_ref = None
cod_label = None                       # (300, 1): código alfabético 0..3
r2_label = r2_onehot = None            # R² medio KFold(5, shuffle=True, random_state=0) con LinearRegression

c_ent, y_ent = ruido["estacion"].to_numpy()[:300], ruido["y"].to_numpy()[:300]
c_prb, y_prb = ruido["estacion"].to_numpy()[300:], ruido["y"].to_numpy()[300:]
r2_ent_ing = r2_prb_ing = r2_ent_oof = r2_prb_oof = None

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert D_completa is not None and D_ref is not None and D_completa.shape == (300, 4) and D_ref.shape == (300, 3), "D_completa debe ser 300×4 (una columna por tipo) y D_ref 300×3 (sin la primera categoría)."
_g = pd.get_dummies(aire["tipo"], dtype=float)[list(CATS)].to_numpy()
assert np.allclose(D_completa, _g) and np.allclose(D_ref, _g[:, 1:]) and D_completa.dtype.kind == "f", "Tu one-hot debe coincidir con pd.get_dummies (columnas en el orden de CATS) y ser de tipo float."
assert rango_completa == 4 and rango_ref == 4, "[1 | D_completa] tiene 5 columnas pero rango 4 (la suma de las indicadoras es la columna de unos: trampa de la variable ficticia); [1 | D_ref] tiene 4 columnas y rango 4."
assert cod_label is not None and cod_label.shape == (300, 1) and set(np.unique(cod_label)) == {0.0, 1.0, 2.0, 3.0} and cod_label[aire["tipo"].to_numpy() == "vial"].min() == 3, "cod_label: industrial=0, rural=1, urbana=2, vial=3, como columna (300, 1)."
_a = cross_val_score(LinearRegression(), cod_label, aire["pm"], cv=cv5, scoring="r2").mean()
_b = cross_val_score(LinearRegression(), D_ref, aire["pm"], cv=cv5, scoring="r2").mean()
assert r2_label is not None and abs(r2_label - _a) < 1e-9 and abs(r2_onehot - _b) < 1e-9, "r2_label y r2_onehot: cross_val_score(LinearRegression(), X, aire['pm'], cv=cv5, scoring='r2').mean() con X = cod_label y X = D_ref."
assert r2_onehot > r2_label + 0.4, "El one-hot debe superar al label encoding por más de 0.4 en R² de validación cruzada: el orden 0-1-2-3 es inventado."
_c = te_ingenuo(np.array(["a", "a", "b"]), np.array([1.0, 3.0, 5.0]), np.array(["a", "b", "z"]))
assert _c is not None and np.allclose(_c, [2.0, 5.0, 3.0]), "te_ingenuo: 'a' → media 2.0, 'b' → 5.0 y la categoría nueva 'z' → media global 3.0."
_o = te_oof(np.array(["a", "a", "b", "a", "b", "a"]), np.array([1.0, 3.0, 5.0, 7.0, 9.0, 11.0]), 2)
assert _o is not None and np.allclose(_o, [7.0, 1.0, 7.0, 1.0, 7.0, 1.0]), "te_oof con k = 2 (pliegue = i % 2): la fila 0 ('a') se codifica con el otro pliegue (filas 1, 3, 5 → 'a' vale 7); la fila 1 con el pliegue 0 ('a' vale 1); la fila 2 ('b', no vista en el otro pliegue) recibe la media global de ese pliegue (7)."
assert r2_ent_ing is not None and r2_ent_ing > 0.25 and r2_prb_ing < 0.0 and r2_ent_ing - r2_prb_ing > 0.5, "El target encoding ingenuo sobre ruido puro debe verse bien en entrenamiento (R² > 0.25) y fallar en prueba (R² < 0)."
assert r2_ent_oof < 0.05 and r2_prb_oof < 0.05, "Con la codificación out-of-fold el R² de entrenamiento debe ser casi 0 (no hay nada que aprender): no se hace ilusiones."
print(f"✅ Reto 1 superado: rango {rango_completa} con 5 columnas (trampa) | R² CV label {r2_label:.2f} frente a one-hot {r2_onehot:.2f} | TE ingenuo: entrenamiento {r2_ent_ing:.2f} → prueba {r2_prb_ing:.2f}; out-of-fold: {r2_ent_oof:.3f} → {r2_prb_oof:.3f}")

---
## Reto 2: Contar excedencias y resistir a los atípicos 🔢

**Contexto:** `conteos` trae, por estación-mes, el número de `excedencias` del límite diario de PM2.5, el nivel de `trafico` (estandarizado), el `tipo` de estación (referencia: `rural`) y los `dias` monitoreados (la **exposición**). Los conteos están **sobredispersos**. Aparte, `(xc, yc)` es una relación lineal de pendiente verdadera 2 contaminada por un sensor descalibrado.

**Enunciado**
1. `X2`: matriz de diseño con columnas constante, `trafico`, `tipo_urbana`, `tipo_vial`. Implementa `irls_poisson(X, y, offset, n_iter)` (enlace log, **con *offset*** $\log(\texttt{dias})$): en cada iteración $\eta=X\beta+\text{offset}$, $\mu=e^{\eta}$, respuesta de trabajo $z=\eta-\text{offset}+(y-\mu)/\mu$ y pesos $W=\mu$, y $\beta\leftarrow(X^\top WX)^{-1}X^\top Wz$. Con `n_iter = 25` y $\beta=0$ de inicio, obtén `beta_p`.
2. `rr` $=e^{\hat\beta}$ (razones de tasas). Con $\hat\mu$ final: `ee_pois` $=\sqrt{\operatorname{diag}[(X^\top WX)^{-1}]}$, `phi` $=\dfrac{1}{n-p}\sum\dfrac{(y_i-\hat\mu_i)^2}{\hat\mu_i}$ (Pearson) y `ee_cuasi` $=\texttt{ee\_pois}\cdot\sqrt{\hat\phi}$.
3. `theil_sen(x, y)` → `(pendiente, intercepto)`: la **mediana de todas las pendientes** $(y_j-y_i)/(x_j-x_i)$ con $i<j$ y $x_j\ne x_i$, e intercepto $\operatorname{mediana}(y)-\text{pendiente}\cdot\operatorname{mediana}(x)$. Guarda `ts_pend` y `mco_pend` (la pendiente de MCO con `np.polyfit`).

**Criterios de aceptación:** `beta_p`, `ee_pois` y `ee_cuasi` coinciden con `sm.GLM` (Poisson, *offset*, `scale='X2'`); `phi` es mucho mayor que 1 (por eso el error estándar de Poisson es demasiado optimista); y Theil-Sen queda más cerca de la pendiente verdadera 2 que MCO.

**Puntaje sugerido:** 22 puntos.

In [ ]:
conteos = generar_conteos()
yc2 = conteos["excedencias"].to_numpy().astype(float)
off = np.log(conteos["dias"].to_numpy().astype(float))
xc, yc = generar_contaminada()


def irls_poisson(X, y, offset, n_iter=25):
    # TODO: IRLS de Poisson con enlace log y offset; devuelve beta
    return None


def theil_sen(x, y):
    # TODO: (mediana de las pendientes por pares, mediana(y) - pendiente * mediana(x))
    return None


# TODO: completa
X2 = None              # (200, 4): constante, trafico, tipo_urbana, tipo_vial (referencia rural)
beta_p = None
rr = None              # exp(beta_p)
ee_pois = phi = ee_cuasi = None
ts_pend = ts_int = mco_pend = None

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert X2 is not None and X2.shape == (200, 4) and np.allclose(X2[:, 0], 1) and np.allclose(X2[:, 1], conteos["trafico"]), "X2 es 200×4: constante, trafico, tipo_urbana, tipo_vial (get_dummies con drop_first=True: la referencia es rural)."
assert np.allclose(X2[:, 2], conteos["tipo"] == "urbana") and np.allclose(X2[:, 3], conteos["tipo"] == "vial"), "Las columnas 2 y 3 son tipo_urbana y tipo_vial."
_g = sm.GLM(yc2, X2, family=sm.families.Poisson(), offset=off).fit()
assert beta_p is not None and np.allclose(beta_p, _g.params, atol=1e-6), "beta_p debe coincidir con sm.GLM Poisson con offset. Revisa: eta = X·beta + offset; z = eta - offset + (y - mu)/mu; W = mu; beta = solve(XᵀWX, XᵀWz); 25 iteraciones desde beta = 0."
assert np.allclose(rr, np.exp(_g.params), atol=1e-6), "rr = np.exp(beta_p): razones de tasas (multiplicativas)."
assert ee_pois is not None and np.allclose(ee_pois, _g.bse, atol=1e-6), "ee_pois = sqrt(diag((XᵀWX)⁻¹)) con W = diag(mu_hat)."
_c = sm.GLM(yc2, X2, family=sm.families.Poisson(), offset=off).fit(scale="X2")
assert phi is not None and abs(phi - _c.scale) < 1e-6 and phi > 2, "phi = Σ (y - mu)²/mu / (n - p) (chi-cuadrado de Pearson sobre sus grados de libertad); con estos datos debe ser mayor que 2."
assert ee_cuasi is not None and np.allclose(ee_cuasi, _c.bse, atol=1e-6) and np.all(ee_cuasi > ee_pois), "ee_cuasi = ee_pois · sqrt(phi): más grande que el de Poisson."
_t = stats.theilslopes(yc, xc)
assert ts_pend is not None and abs(ts_pend - _t[0]) < 1e-9, "ts_pend = mediana de (y_j - y_i)/(x_j - x_i) para todos los pares i < j con x distintos."
assert abs(ts_int - (np.median(yc) - _t[0] * np.median(xc))) < 1e-9, "ts_int = mediana(y) - pendiente · mediana(x)."
assert mco_pend is not None and abs(mco_pend - np.polyfit(xc, yc, 1)[0]) < 1e-9 and abs(ts_pend - 2) < abs(mco_pend - 2) and abs(ts_pend - 2) < 0.35, "Theil-Sen debe quedar a menos de 0.35 de la pendiente verdadera 2 y más cerca que MCO (np.polyfit)."
print(f"✅ Reto 2 superado: razón de tasas del tráfico {rr[1]:.3f} por desviación estándar | φ = {phi:.2f} → error estándar del tráfico {ee_pois[1]:.4f} (Poisson) frente a {ee_cuasi[1]:.4f} (cuasi) | pendiente verdadera 2: Theil-Sen {ts_pend:.2f}, MCO {mco_pend:.2f}")

---
## Reto 3: Evaluar sin engañarse 🧪

**Contexto:** `Xs, ys` (80 observaciones) dependen solo de las **2 primeras** columnas; las otras 38 son ruido. Usa las primeras 50 filas como entrenamiento (`tr`) y las últimas 30 como prueba (`te`). Aparte, `X_pura, y_pura` son 150 variables y una respuesta **totalmente independientes**.

**Enunciado**
1. Funciones `r2(y, yhat)`, `r2_aj(r2, n, p)` ($p$ = número de predictores sin la constante) y `rmse(y, yhat)`. Ajusta MCO con 2 predictores y con 37 (las primeras columnas, **con** intercepto, `LinearRegression`) y guarda `r2_ent_2`, `r2_prb_2`, `r2_ent_37`, `r2_prb_37`, y `r2aj_ent_37` ($R^2$ ajustado en entrenamiento del modelo grande).
2. `aic_bic(sce, n, k)` → `(AIC, BIC)` con la verosimilitud gaussiana en el máximo, $\ell=-\frac n2\left[\ln(2\pi)+\ln(SCE/n)+1\right]$, $AIC=-2\ell+2k$, $BIC=-2\ell+k\ln n$ y $k$ = número de coeficientes **incluida la constante**. Calcula `aic_2`, `bic_2`, `aic_37`, `bic_37` sobre el entrenamiento.
3. `cv_mse(X, y, k)`: MSE medio de validación cruzada con pliegues contiguos (`np.array_split(np.arange(n), k)`) y MCO con intercepto (`np.linalg.lstsq`) en cada pliegue. Guarda `cv5` $=$ `cv_mse(Xs[tr, :2], ys[tr], 5)`.
4. `loo_atajo(X, y)`: error cuadrático medio *leave-one-out* **sin reajustar**, con $\frac1n\sum\left(\frac{e_i}{1-h_{ii}}\right)^2$ ($h_{ii}$ de la matriz sombrero del modelo con intercepto). Guarda `loo` para `Xs[tr, :2]`.
5. **Fuga.** Con `cv5k = KFold(5, shuffle=True, random_state=0)`: `r2_fuga` es el $R^2$ medio de validación cruzada tras elegir las **10 columnas más correlacionadas con `y_pura` usando TODAS las filas**; `r2_pipe` hace lo mismo con la selección **dentro** de cada pliegue (`Pipeline` con `SelectKBest(f_regression, k=10)` y `LinearRegression`).

**Criterios de aceptación:** las métricas coinciden con `scikit-learn`/`statsmodels`; el modelo de 37 variables brilla en entrenamiento y **se hunde** en prueba, y el BIC prefiere el de 2; `cv5` coincide con `cross_val_score`; `loo_atajo` coincide con el *leave-one-out* explícito; y con ruido puro `r2_fuga` es positivo mientras `r2_pipe` es negativo.

**Puntaje sugerido:** 22 puntos.

In [ ]:
Xs, ys = generar_sobreajuste()
tr, te = np.arange(50), np.arange(50, 80)
X_pura, y_pura = generar_ruido_puro()
cv5k = KFold(5, shuffle=True, random_state=0)


def r2(y, yhat):
    # TODO: 1 - SCE/SCT
    return None


def r2_aj(r2_, n, p):
    # TODO: R² ajustado con p predictores (sin contar la constante)
    return None


def rmse(y, yhat):
    # TODO: raíz del error cuadrático medio
    return None


def aic_bic(sce, n, k):
    # TODO: (AIC, BIC) con la verosimilitud gaussiana en el máximo; k incluye la constante
    return None


def cv_mse(X, y, k):
    # TODO: MSE medio de k pliegues contiguos (np.array_split) con MCO + intercepto
    return None


def loo_atajo(X, y):
    # TODO: ECM leave-one-out con el atajo de la matriz sombrero (sin reajustar n veces)
    return None


# TODO: completa
r2_ent_2 = r2_prb_2 = r2_ent_37 = r2_prb_37 = r2aj_ent_37 = None
aic_2 = bic_2 = aic_37 = bic_37 = None
cv5 = loo = None
r2_fuga = r2_pipe = None

In [ ]:
# Autoverificación del Reto 3 (no modificar)
_yy = np.array([1.0, 2.0, 4.0, 3.0])
_hh = np.array([1.5, 2.0, 3.0, 3.5])
assert r2(_yy, _hh) is not None and abs(r2(_yy, _hh) - r2_score(_yy, _hh)) < 1e-12, "r2 = 1 - Σ(y - ŷ)² / Σ(y - ȳ)²."
assert abs(rmse(_yy, _hh) - math.sqrt(mean_squared_error(_yy, _hh))) < 1e-12, "rmse = sqrt(media((y - ŷ)²))."
assert r2_aj(0.5, 20, 3) is not None and abs(r2_aj(0.5, 20, 3) - (1 - 0.5 * 19 / 16)) < 1e-12, "R² ajustado = 1 - (1 - R²)(n - 1)/(n - p - 1)."
_m2 = LinearRegression().fit(Xs[tr, :2], ys[tr])
_m37 = LinearRegression().fit(Xs[tr, :37], ys[tr])
assert r2_ent_2 is not None and abs(r2_ent_2 - _m2.score(Xs[tr, :2], ys[tr])) < 1e-9 and abs(r2_prb_2 - _m2.score(Xs[te, :2], ys[te])) < 1e-9, "r2_ent_2 y r2_prb_2: R² del modelo de 2 predictores en entrenamiento y en prueba."
assert abs(r2_ent_37 - _m37.score(Xs[tr, :37], ys[tr])) < 1e-9 and abs(r2_prb_37 - _m37.score(Xs[te, :37], ys[te])) < 1e-9, "r2_ent_37 y r2_prb_37: igual para el modelo de 37 predictores."
assert r2_ent_37 > r2_ent_2 and r2_prb_37 < r2_prb_2 - 1.0, "El modelo grande debe MEJORAR en entrenamiento y hundirse en prueba (más de 1 punto de R² por debajo del pequeño)."
assert abs(r2aj_ent_37 - (1 - (1 - r2_ent_37) * 49 / 12)) < 1e-9 and r2aj_ent_37 < r2_ent_37, "r2aj_ent_37 = 1 - (1 - R²)(n - 1)/(n - p - 1) con n = 50 y p = 37."
_a2, _a37 = sm.OLS(ys[tr], sm.add_constant(Xs[tr, :2])).fit(), sm.OLS(ys[tr], sm.add_constant(Xs[tr, :37])).fit()
assert aic_2 is not None and abs(aic_2 - _a2.aic) < 1e-7 and abs(bic_2 - _a2.bic) < 1e-7, "AIC = -2ℓ + 2k y BIC = -2ℓ + k·ln(n), con ℓ = -n/2·[ln(2π) + ln(SCE/n) + 1] y k = 3 (dos pendientes y la constante)."
assert abs(aic_37 - _a37.aic) < 1e-6 and abs(bic_37 - _a37.bic) < 1e-6 and bic_2 < bic_37, "Con k = 38 el modelo grande debe coincidir con statsmodels y tener MAYOR BIC que el de 2 predictores."
_cvr = -cross_val_score(LinearRegression(), Xs[tr, :2], ys[tr], cv=KFold(5), scoring="neg_mean_squared_error").mean()
assert cv5 is not None and abs(cv5 - _cvr) < 1e-9, "cv5: pliegues contiguos (np.array_split), reajusta MCO con intercepto en cada entrenamiento y promedia el MSE de cada pliegue de prueba."
_loo = -cross_val_score(LinearRegression(), Xs[tr, :2], ys[tr], cv=LeaveOneOut(), scoring="neg_mean_squared_error").mean()
assert loo is not None and abs(loo - _loo) < 1e-9, "loo = media((e_i / (1 - h_ii))²), con e = y - Hy y H = A(AᵀA)⁻¹Aᵀ, A con columna de unos."
assert r2_fuga is not None and r2_fuga > 0.05, f"Eligiendo las 10 mejores columnas con TODAS las filas, la validación cruzada de ruido puro engañosamente da R² > 0 (tú: {r2_fuga})."
assert r2_pipe is not None and r2_pipe < -0.1, f"Con la selección DENTRO de cada pliegue (Pipeline) el R² de ruido puro debe ser negativo (tú: {r2_pipe})."
print(f"✅ Reto 3 superado: 2 variables R² {r2_ent_2:.2f} → {r2_prb_2:.2f}; 37 variables {r2_ent_37:.2f} → {r2_prb_37:.2f} (ajustado {r2aj_ent_37:.2f}) | BIC {bic_2:.1f} frente a {bic_37:.1f} | CV5 {cv5:.2f} y LOO {loo:.2f} | ruido puro: con fuga R² = {r2_fuga:.2f}, con pipeline {r2_pipe:.2f}")

---
## Reto 4: De los predios al informe, con dos bibliotecas 🧰

**Contexto:** `predios` trae el `indice` de calidad de 400 predios con tres variables numéricas (`area`, `altitud` —con valores faltantes—, `pendiente`) y dos categóricas (`uso`, `vereda`). Usa las primeras 300 filas como entrenamiento (`train`) y las últimas 100 como prueba (`test`). Vas a armar un único flujo que no filtre información, evaluarlo una sola vez en prueba y comprobar que `scikit-learn` y `statsmodels` dan **el mismo modelo**.

**Enunciado**
1. `pipe`: `Pipeline` con un `ColumnTransformer` (`sparse_threshold=0`) y `LinearRegression`. Numéricas (`NUM`): imputación por **mediana** y `StandardScaler`; categóricas (`CAT`): `OneHotEncoder(drop="first", handle_unknown="ignore")`. Ajústalo con `train`. Guarda `rmse_test` (RMSE en `test`).
2. `pred_nueva`: predicción de `pipe` para `fila_nueva` (una vereda **nunca vista** y la altitud **faltante**) y `ic_rmse`: IC de 95 % *bootstrap* percentil del RMSE de prueba (`B = 2000` remuestreos de las 100 filas de prueba con reemplazo; `np.random.default_rng(0)`).
3. **Puente:** `Xt = pipe[:-1].transform(train[NUM + CAT])` y `res_sm = sm.OLS(ytr, sm.add_constant(Xt)).fit()`. Guarda `coef_sm` (`res_sm.params`) e `ic_area` (IC de 95 % del coeficiente de `area`, que es la primera columna de `Xt`).
4. `r2_cv`: $R^2$ medio de `cross_val_score(pipe, train[NUM + CAT], ytr, cv=KFold(5, shuffle=True, random_state=0), scoring="r2")` (el *pipeline* completo se reajusta en cada pliegue, así que no hay fuga).

**Criterios de aceptación:** `Xt` tiene 10 columnas; `pred_nueva` es finita **y** igual a predecir la misma fila con la vereda de referencia y la altitud en la mediana del entrenamiento; `ic_rmse` contiene a `rmse_test`; `coef_sm` coincide con el intercepto y los coeficientes de `LinearRegression`; y el coeficiente de `area` cae dentro de su IC.

**Puntaje sugerido:** 19 puntos.

In [ ]:
predios = generar_predios()
NUM, CAT = ["area", "altitud", "pendiente"], ["uso", "vereda"]
train, test = predios.iloc[:300], predios.iloc[300:]
ytr, yte = train["indice"].to_numpy(), test["indice"].to_numpy()
fila_nueva = pd.DataFrame([{"area": 3.0, "altitud": np.nan, "pendiente": 12.0, "uso": "cultivo", "vereda": "Zarzal"}])

# TODO: completa
pipe = None            # Pipeline(ColumnTransformer + LinearRegression), ajustado con train
rmse_test = None
pred_nueva = None
ic_rmse = None         # (percentil 2.5, percentil 97.5) del RMSE por bootstrap sobre test
Xt = None              # pipe[:-1].transform(train[NUM + CAT]), denso
res_sm = coef_sm = ic_area = None
r2_cv = None

In [ ]:
# Autoverificación del Reto 4 (no modificar)
assert pipe is not None and hasattr(pipe, "predict") and hasattr(pipe, "named_steps") and len(pipe.steps) == 2, "pipe debe ser un Pipeline de dos pasos: el ColumnTransformer y LinearRegression, ya ajustado con train."
_pt = pipe.predict(test[NUM + CAT])
assert rmse_test is not None and abs(rmse_test - math.sqrt(mean_squared_error(yte, _pt))) < 1e-9, "rmse_test = sqrt(MSE) de pipe.predict(test[NUM + CAT]) frente a yte."
assert 2 < rmse_test < 8, f"El RMSE de prueba debería rondar el ruido verdadero (σ = 4); el tuyo es {rmse_test:.2f}. Revisa el pipeline (mediana, escalador, one-hot con drop='first')."
assert Xt is not None and np.ndim(Xt) == 2 and Xt.shape == (300, 10) and not hasattr(Xt, "toarray"), "Xt debe ser DENSO y de 300×10 (3 numéricas + 2 de uso + 5 de vereda): usa sparse_threshold=0 en el ColumnTransformer."
_ref = pd.DataFrame([{"area": 3.0, "altitud": float(np.nanmedian(train["altitud"])), "pendiente": 12.0, "uso": "cultivo", "vereda": "Alisos"}])
assert pred_nueva is not None and np.isfinite(pred_nueva) and abs(pred_nueva - pipe.predict(_ref)[0]) < 1e-8, "pred_nueva debe ser finita y IGUAL a predecir con la altitud en la mediana del entrenamiento y la vereda de referencia (Alisos): imputer con mediana y OneHotEncoder(drop='first', handle_unknown='ignore')."
assert ic_rmse is not None and len(ic_rmse) == 2 and ic_rmse[0] < rmse_test < ic_rmse[1] and 0.05 * rmse_test < ic_rmse[1] - ic_rmse[0] < 1.5 * rmse_test, "ic_rmse = percentiles 2.5 y 97.5 del RMSE de 2000 remuestreos (con reemplazo) de las 100 filas de prueba; debe contener a rmse_test."
_lin = pipe[-1]
assert coef_sm is not None and len(coef_sm) == 11 and abs(coef_sm[0] - _lin.intercept_) < 1e-6 and np.allclose(coef_sm[1:], _lin.coef_, atol=1e-6), "statsmodels y scikit-learn deben dar el MISMO modelo: coef_sm[0] = intercept_ y coef_sm[1:] = coef_. Recuerda sm.add_constant(Xt)."
assert ic_area is not None and ic_area[0] < _lin.coef_[0] < ic_area[1] and ic_area[1] - ic_area[0] > 0, "ic_area = res_sm.conf_int()[1] (la columna 1 es `area`, la primera tras la constante) y debe contener al coeficiente de scikit-learn."
_cv = cross_val_score(pipe, train[NUM + CAT], ytr, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
assert r2_cv is not None and abs(r2_cv - _cv) < 1e-9 and 0.3 < r2_cv < 0.95, "r2_cv: cross_val_score del pipeline COMPLETO sobre train con KFold(5, shuffle=True, random_state=0)."
print(f"✅ Reto 4 superado: RMSE de prueba {rmse_test:.2f} (IC95 {ic_rmse[0]:.2f}-{ic_rmse[1]:.2f}) | R² CV {r2_cv:.2f} | predicción para la fila nueva {pred_nueva:.1f} | scikit-learn = statsmodels (11 coeficientes) | IC95 del área ({ic_area[0]:.2f}, {ic_area[1]:.2f})")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados, con tus cifras. Debe responder:

- Con `r2_label` y `r2_onehot`: ¿por qué un *label encoding* alfabético le **inventa** un orden al modelo y cuándo sí sería razonable un código ordinal? ¿Qué enseñan `rango_completa` y `rango_ref` sobre la trampa de la variable ficticia?
- Con `r2_ent_ing`, `r2_prb_ing` y `r2_prb_oof`: ¿por qué el *target encoding* ingenuo parece funcionar en entrenamiento sobre ruido puro y qué hace el *out-of-fold* para impedirlo?
- Con `phi`: ¿qué le pasa a los errores estándar de Poisson cuando hay sobredispersión, a quién le conviene creerlos y qué alternativas tienes (cuasi-Poisson, binomial negativa)? ¿Qué ganó Theil-Sen frente a MCO y qué costó (eficiencia)?
- Con `r2_ent_37`, `r2_prb_37`, `bic_2` y `bic_37`: ¿qué criterio de los tres (R² de entrenamiento, BIC, validación cruzada) detecta el sobreajuste y por qué el $R^2$ de entrenamiento nunca baja al agregar variables? ¿Qué demuestra la diferencia entre `r2_fuga` y `r2_pipe`?
- Con `rmse_test`, `ic_rmse` y la comparación `coef_sm` frente a `scikit-learn`: ¿para qué usarías cada biblioteca (inferir o predecir)? ¿Qué te dice el IC del RMSE sobre cuánto confiar en un solo conjunto de prueba?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: one-hot desde NumPy, trampa de la variable ficticia, label frente a one-hot y target encoding ingenuo frente a *out-of-fold* | 22 |
| Reto 2: IRLS de Poisson con *offset*, sobredispersión (cuasi-Poisson) y Theil-Sen frente a MCO | 22 |
| Reto 3: $R^2$, $R^2$ ajustado, AIC/BIC, *k-fold*, atajo *leave-one-out* y fuga al seleccionar variables | 22 |
| Reto 4: *pipeline* con `ColumnTransformer`, IC *bootstrap* del RMSE y puente `statsmodels` / `scikit-learn` | 19 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen ✅.
- [ ] Mi selección de variables y mi *target encoding* ocurren **dentro** de la validación cruzada, no antes.
- [ ] Reporté la incertidumbre (intervalos) y no solo un número puntual.
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Estadística, Análisis y Representación de Datos</i>
  </p>
</div>